# Module 2: Error analysis and synthetic test data

In this notebook you'll:

1. Configure the Phoenix Python client to pull traces in the same way as module 1
2. Run an expanded set of hand-written queries against the same agent
3. Categorize the failures by root cause and severity
4. Save the labeled failures as a Phoenix dataset
5. Generate synthetic test queries with an LLM and surface new failure modes
6. Grow the dataset into a proper labeled benchmark

This notebook assumes you've already run the module 1 notebook. It uses the same project (`dlai-ecommerce-agent`) and the same agent definition.

## Table of Contents
- [Imports](#Imports)
- [1 - Configure the environment and Phoenix](#1---configure-the-environment-and-phoenix)
    - **[Exercise 1 - client_init](#exercise-1---client_init)**
- [2 - Collect and inspect agent traces](#2---collect-and-inspect-agent-traces)
    - **[Exercise 2 - get_parent_spans](#exercise-2---get_parent_spans)**
- [3 - Build an evaluation test suite](#3---build-an-evaluation-test-suite)
- [4 - Run the evaluation and review results](#4---run-the-evaluation-and-review-results)
- [5 - Label failures and create a benchmark dataset](#5---label-failures-and-create-a-benchmark-dataset)
    - **[Exercise 3 - label_traces](#exercise-3---label_traces)**
- [6 - Explore labeled examples](#6---explore-labeled-examples)
    - **[Exercise 4 - create_failure_dataset](#exercise-4---create_failure_dataset)**

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, make sure you ran all of the unit tests and then click on the `Submit assignment` button on the top right of the page.
---

<a id="imports"></a>
## Imports

This section loads the libraries needed throughout the notebook.

These packages provide the functionality required to:
- Connect to Phoenix and retrieve agent traces.
- Manipulate and analyze trace data using pandas.
- Run evaluation queries against the agent.
- Use helper functions and testing utilities provided for this module.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import random
import pandas as pd
import phoenix as px

from phoenix.client import Client
from tqdm.auto import tqdm
from anthropic import Anthropic

import utils
import unittests

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dlai-ecommerce-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



<a id="1---configure-the-environment-and-phoenix"></a>
## 1 - Configure the environment and Phoenix

**Set your API keys**

The agent needs credentials for two services: 
- Anthropic, to call the model
- Phoenix, to receive the traces.

You need to set these up as environment variables for the SDKs to work correctly.

You also need to define a collector endpoint which is the address traces get sent to. Without it, the agent still runs but nothing lands in Phoenix.

Within the DLAI platform these are already set up for you but if you are working outside it make sure to define three variables yourself:

| Variable | Where to get it |
|---|---|
| `ANTHROPIC_API_KEY` | [console.anthropic.com](https://console.anthropic.com/settings/keys) → Settings → API Keys |
| `PHOENIX_API_KEY` | [app.phoenix.arize.com](https://app.phoenix.arize.com/login) → your Space → Settings |
| `PHOENIX_COLLECTOR_ENDPOINT` | Same Settings page, listed as your Hostname |

You can do this by creating a cell and placing this code along with the correct information:
```python
import os

os.environ["ANTHROPIC_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "your-actual-endpoint-here"
```

Initialize Phoenix by running the following cell:

In [2]:
# Launch Phoenix
px.launch_app()

# Show the true Phoenix URL
utils.show_phoenix_link()

boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


Could not resolve 5 span IDs to database records. Examples will be created without span links.


**NOTE:** Within the utils file, the database is created and the agent is set up in the exact manner as module 1. If you want to remember how this code looks like you can do so by looking at `utils.py` but it is abstracted here so you can focus on the contents of this module.

In [3]:
# Check the tables defined in Module 1
utils.display_tables()

### `products` (20 rows)
Catalog product facts, including descriptions and care instructions.

,id,name,category,color,fabric,description,price,available,returnable,final_sale,care
0,TEE-001,Classic Tee,tops,white,combed cotton,100% combed cotton crew-neck t-shirt. Pre-shru...,24.99,1,1,0,
1,SHO-001,Trail Runner,shoes,black,synthetic mesh,Lightweight trail running shoe with rock plate...,89.99,1,1,0,
2,SHO-002,Court Shoe,shoes,white,canvas and rubber,Low-profile court shoe with gum rubber outsole...,74.99,1,1,0,
3,SHO-003,Slip-On Canvas,shoes,navy,canvas,Canvas slip-on shoe with elastic side panels. ...,55.00,0,1,0,
4,OUT-001,Denim Jacket,outerwear,blue,cotton denim,Classic trucker-style denim jacket. Button fro...,98.00,1,1,0,
5,OUT-002,Raincoat,outerwear,yellow,waterproof nylon,Waterproof hooded raincoat with sealed seams a...,125.00,1,1,0,
6,OUT-003,Fleece Pullover,outerwear,heather gray,polyester fleece,Quarter-zip midweight fleece pullover. Chest p...,68.00,1,1,0,
7,TOP-002,Oxford Shirt,tops,white,Oxford cotton,Button-down Oxford cloth shirt with chest pock...,68.00,1,1,0,
8,TOP-003,Hooded Sweatshirt,tops,gray,cotton blend,Midweight cotton-blend hoodie with kangaroo po...,54.00,1,1,0,
9,TOP-004,Crew Neck Sweater,tops,cream,merino wool,Merino wool crew-neck sweater. Machine washabl...,72.00,1,1,0,Machine washable on wool cycle.


### `product_colors` (54 rows)
All color options for each product.

,product_id,color
0,TEE-001,white
1,TEE-001,black
2,TEE-001,navy
3,TEE-001,heather gray
4,SHO-001,black
5,SHO-001,gray
6,SHO-002,white
7,SHO-002,black
8,SHO-003,navy
9,SHO-003,gray


### `product_sizes` (72 rows)
All size options for each product.

,product_id,size
0,TEE-001,XS
1,TEE-001,S
2,TEE-001,M
3,TEE-001,L
4,TEE-001,XL
...,...,...
67,ACC-004,one size
68,ACC-005,M
69,ACC-005,L
70,ACC-006,M


... (14 outputs left out)

### `reservations` (0 rows)
Pickup reservations created while the agent is running.

_No rows yet._

### `exchange_requests` (0 rows)
Exchange requests created while the agent is running.

_No rows yet._

Set up the client. This is important in case you run the notebook multiple times. It ensures that traces associated with this project are reset before each run, preventing results from previous executions from affecting the current evaluation.

<a id="exercise-1---client_init"></a>
### Exercise 1 - client_init

**Your Task:**

Your task is to initialize a Phoenix client and ensure that the project starts in a clean state. You need to implement the missing code in three steps:

* **Instantiate the Phoenix client**:
    * Create a new Phoenix client instance using the imported `Client` class and store it in `phoenix_client`.

* **Check if the project already exists**:
    * Use the client to retrieve the project specified by `project_name`.
    * This check helps avoid conflicts when restarting the kernel or rerunning the notebook.

* **Delete the project if it exists**:
    * If the project is found, delete it so that a fresh project can be created during the current run.
    * If the project does not exist, simply continue execution without raising an error.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

* For `phoenix_client`: Instantiate the client with `Client()`.
* To check for an existing project: Call `phoenix_client.projects.get(project_name=project_name)`.
* If the project does not exist: Use `pass` inside the `except` block.
* If the project exists: Delete it with `phoenix_client.projects.delete(project_name=project_name)`.

</details>

In [4]:
# GRADED FUNCTION: client_init

def client_init(project_name="dlai-ecommerce-agent"):

    ### START CODE HERE ###

    # Instantiate the client imported at the top
    phoenix_client = Client()

    # Check if the project already exists (this is useful in case you restart the kernel or want a fresh project)
    try: 
        phoenix_client.projects.get(project_name=project_name)
    except: 
        # Pass if it doesn't exist
        pass
    else: 
        # Delete if it does exist
        phoenix_client.projects.delete(project_name=project_name)

    ### END CODE HERE ###

    return phoenix_client

In [5]:
phoenix_client = client_init()

print(f"phoenix_client has correct type?: {isinstance(phoenix_client, Client)}")

phoenix_client has correct type?: True


**Expected Output:**

```
phoenix_client has correct type?: True
```

In [6]:
# Test your code!
unittests.test_client_init(client_init)

 All tests passed!


<a id="2---collect-and-inspect-agent-traces"></a>
## 2 - Collect and inspect agent traces

A simple interaction is used as a sanity check before running larger evaluations. Besides confirming that the agent responds correctly, this execution creates a trace that can be inspected through Phoenix.

In [7]:
response = utils.ask_agent("hello")
print("ANSWER:", response)

ANSWER: Hello! 👋 Welcome to Northwind Apparel! 

I'm here to help you with:
- **Finding apparel** – Search our catalog by style, color, size, price, and more
- **Checking inventory** – See what's in stock
- **Shopping** – Create carts and reserve items
- **Orders** – Look up tracking, process exchanges, and handle refunds
- **Coupons** – Apply discount codes to your cart

What can I help you with today?


Running one query at a time is useful for exploration, but evaluation requires many interactions. This helper function streamlines the process by sending a list of queries to the agent and displaying shortened versions of the responses.

In [8]:
def run_batch(queries):
    # Iterates through the query list sequentially, updating a progress bar for each item
    for q in tqdm(queries, desc="Processing queries"):
        response = utils.ask_agent(q)
        truncated = response[:150] + ("..." if len(response) > 150 else "")
        print(f"Q: {q}\nA: {truncated}\n{'-' * 60}")

A couple of sample questions are sent to the agent to create initial trace data. These examples provide a quick way to verify that executions are being recorded before moving to the full evaluation set.

In [9]:
response = run_batch(["hi", "do you have shoes?"])
response

Processing queries:   0%|          | 0/2 [00:00<?, ?it/s]

Q: hi
A: Hello! 👋 Welcome to Northwind Apparel! 

I'm here to help you with:
- **Finding apparel** – search our catalog by style, color, size, price, and more
...
------------------------------------------------------------
Q: do you have shoes?
A: Yes! We have 3 shoes available:

1. **Trail Runner** (SHO-001) - $89.99
   - Lightweight trail running shoe with rock plate and aggressive lug pattern...
------------------------------------------------------------


<a id="exercise-2---get_parent_spans"></a>
### Exercise 2 - get_parent_spans

**Your Task:**

Your task is to retrieve span data from Phoenix and extract only the parent spans (spans without a parent). You need to implement the missing code in four steps:

* **Retrieve all spans**:
    * Use the Phoenix client to fetch all spans for the specified project as a pandas DataFrame.

* **Filter for parent spans**:
    * Select only spans whose `parent_id` is missing (`NaN`), indicating that they are top-level spans.
    * Create a copy of the filtered DataFrame to avoid modifying the original data.

* **Check that parent spans exist**:
    * Verify that the resulting DataFrame is not empty before performing any column transformations.

* **Rename input and output columns**:
    * Rename `"attributes.input.value"` to `"input"`.
    * Rename `"attributes.output.value"` to `"output"`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

* For `all_spans`: Call `client.spans.get_spans_dataframe(project_identifier=project_name)`.
* For `parent_spans`: Filter with `all_spans[all_spans["parent_id"].isna()].copy()`.
* To check that spans exist: Use `if not parent_spans.empty:`.
* To rename columns: Call

```python
parent_spans = parent_spans.rename(
    columns={
        "attributes.input.value": "input",
        "attributes.output.value": "output",
    },
)
```

</details>

In [10]:
# GRADED FUNCTION: get_parent_spans

def get_parent_spans(client, project_name):
    ### START CODE HERE ###

    # Get spans from Phoenix client in dataframe format
    all_spans = client.spans.get_spans_dataframe(project_identifier=project_name)

    # Get spans without a parent (copy original df)
    parent_spans = all_spans[all_spans["parent_id"].isna()].copy()

    # Check that the spans are not empty
    if not parent_spans.empty:
        # Rename columns
        parent_spans = parent_spans.rename(
            columns={ 
                "attributes.input.value": "input",
                "attributes.output.value": "output",
            }, 
        ) 

    ### END CODE HERE ###

        # Keep relevant columns
        parent_spans = parent_spans[["input", "output", "context.span_id"]]


    return parent_spans

In [11]:
# This should show the dummy queries you have used so far
parent_spans = get_parent_spans(phoenix_client, "dlai-ecommerce-agent")
print(f"Top-level agent runs: {len(parent_spans)}")
parent_spans.head()

Top-level agent runs: 2


,input,output,context.span_id
context.span_id,,,
ad6124bfb62d8094,hi,Hello! 👋 Welcome to Northwind Apparel! \n\nI'm...,ad6124bfb62d8094
524dbc2cf683bfaa,hello,Hello! 👋 Welcome to Northwind Apparel! \n\nI'm...,524dbc2cf683bfaa


**Expected Output:**

**Note:** The context.span_id values are examples and may be different when you run the code.

```Top-level agent runs: 3```

| context.span_id | input | output | context.span_id |
|-----------------|-------|--------|-----------------|
|77a2e9d7c453c457|	do you have shoes?	|Yes! We have 3 shoes available:\n\n1. **Trail ...	| 77a2e9d7c453c457
| 124a6b14681b56e0 | hi | Hello! 👋 Welcome to Northwind Apparel! \n\nI'm... | 124a6b14681b56e0 |
| 74a587374dba6b2a | hello | Hello! Welcome to Northwind Apparel! 👋 \n\nI'm... | 74a587374dba6b2a |


In [12]:
# Test your code!
unittests.test_get_parent_spans(get_parent_spans)

Failed test case: Incorrect number of spans for dlai-ecommerce-agent project.
Expected: at least 3 (you ran 3 dummy queries)
Got: 2




The evaluation should start with a clean Phoenix project so that only the relevant runs are included in the analysis. Resetting the project removes previous traces that could interfere with the results.

In [13]:
# Initialize a client with no runs
phoenix_client = client_init()
parent_spans = get_parent_spans(phoenix_client, "dlai-ecommerce-agent")
print(f"Top-level agent runs: {len(parent_spans)}")

Top-level agent runs: 0


<a id="3---build-an-evaluation-test-suite"></a>
## 3 - Build an evaluation test suite

A good evaluation set should not only test whether the agent can answer common questions, but also explore situations where it may fail.

The following queries are organized around different failure patterns, such as unsupported assumptions, hallucinations, consistency issues, and requests outside the agent's scope.

* **Baseline Queries:** Standard, straightforward questions about active catalog items to verify basic retrieval and response accuracy.
* **Hallucination Bait:** Asking for non-existent items to ensure the agent admits it doesn't carry them rather than inventing fake products.
* **Faithfulness Bait:** Specific attribute questions where the model might make unsupported assumptions or extrapolate beyond what product descriptions state.
* **Scope Violations:** Out-of-bounds policy or store questions to test whether the agent correctly identifies its boundaries versus answering unsupported topics.
* **Consistency Tests:** Repeated identical queries to verify output stability and ensure tone or reasoning doesn't drift.
* **Multi-Part Queries:** Complex requests combining multiple constraints (e.g., price + category + color) to test reasoning and multi-constraint filtering.
* **Odd Phrasing:** Unstructured, informal, or typo-heavy user inputs to test robustness against n

In [14]:
BASELINE_QUERIES = [
    "What's the price of the Wool Beanie?",
    "Do you have a leather wallet?",
    "What colors does the Hooded Sweatshirt come in?",
    "Tell me about the Trail Runner.",
    "Is the Slip-On Canvas in stock right now?",
]

HALLUCINATION_BAIT = [
    "Do you have the AeroFlow running shorts?",
    "I'm looking for your linen blazer.",
    "Do you sell winter boots?",
    "I want to buy the Performance Polo in size large.",
]

FAITHFULNESS_BAIT = [
    "How tall do the cargo shorts come up? Are they above the knee?",
    "What's the difference between the Trail Runner and the Court Shoe?",
    "Can the canvas backpack hold a 17-inch laptop?",
    "Is the Oxford Shirt good for hot weather?",
]

SCOPE_VIOLATIONS = [
    "How long does shipping to California take?",
    "Can I exchange a shirt I bought last month?",
    "What size pants am I if I'm normally a 32-inch waist?",
    "Do you have a physical store location?",
]

CONSISTENCY_TESTS = [
    "What should I wear to a casual outdoor wedding?",
    "What should I wear to a casual outdoor wedding?",
    "What should I wear to a casual outdoor wedding?",
    "Help me put together a back-to-school outfit.",
    "Help me put together a back-to-school outfit.",
    "Help me put together a back-to-school outfit.",
]

MULTI_PART = [
    "I want a pair of pants and a matching belt. What goes with the chinos?",
    "I need both gym clothes and something for the office. Suggestions?",
    "Do you have anything in olive that's under $70?",
    "What's the cheapest top and shoe combo you have?",
]

ODD_PHRASING = [
    "got any nice flannel?",
    "Wallet??",
    "looking for shoes for my husband size 11 not too expensive",
    "tell me your softest shirt thanks",
    "i need stuff to wear in the rain its been raining for a week",
]

ALL_QUERIES = (
    BASELINE_QUERIES
    + HALLUCINATION_BAIT
    + FAITHFULNESS_BAIT
    + SCOPE_VIOLATIONS
    + CONSISTENCY_TESTS
    + MULTI_PART
    + ODD_PHRASING
)
print(f"Total queries to run: {len(ALL_QUERIES)}")

Total queries to run: 32


<a id="4---run-the-evaluation-and-review-results"></a>
## 4 - Run the evaluation and review results

The complete evaluation suite is now executed against the agent. Each interaction is captured as a trace, creating the raw data needed for later analysis.

In [15]:
run_batch(ALL_QUERIES)

Processing queries:   0%|          | 0/32 [00:00<?, ?it/s]

Q: What's the price of the Wool Beanie?
A: The **Wool Beanie** (ACC-002) is **$22.00**. It's a cuffed merino-wool beanie available in black, gray, navy, and oatmeal, and it's one size fits most...
------------------------------------------------------------
Q: Do you have a leather wallet?
A: Yes! We have a **Leather Wallet** (ACC-004) available:

- **Price:** $45.00
- **Description:** Bifold leather wallet with six card slots and bill comp...
------------------------------------------------------------
Q: What colors does the Hooded Sweatshirt come in?
A: The **Hooded Sweatshirt** (TOP-003) comes in three colors:
- **Black**
- **Forest Green**
- **Gray**

It's a midweight cotton-blend hoodie with a kang...
------------------------------------------------------------
Q: Tell me about the Trail Runner.
A: Great! Here's what we have:

**Trail Runner** (SHO-001)
- **Price:** $89.99
- **Category:** Shoes
- **Description:** Lightweight trail running shoe wi...
------------------------------

After the evaluation finishes, the collected traces are loaded from Phoenix. The dataframe now contains the complete set of user queries and generated responses that will be reviewed.

In [16]:
# Now the df should have a row per query
parent_spans = get_parent_spans(phoenix_client, "dlai-ecommerce-agent")
print(f"Top-level agent runs: {len(parent_spans)}")
parent_spans.head()

Top-level agent runs: 31


,input,output,context.span_id
context.span_id,,,
244c1ee9fff3aaa1,tell me your softest shirt thanks,"Based on our search, I'd recommend the **Class...",244c1ee9fff3aaa1
94464a6ad25f992e,looking for shoes for my husband size 11 not t...,Great! I found 3 shoes in size 11 under $100:\...,94464a6ad25f992e
ce7eb985a4b66f22,Wallet??,I'm not sure what you're looking for! Are you ...,ce7eb985a4b66f22
07cc721984962d68,got any nice flannel?,Yes! We have a **Flannel Shirt** (TOP-005) tha...,07cc721984962d68
44edef8bd76d82ea,What's the cheapest top and shoe combo you have?,"Based on our available inventory, the cheapest...",44edef8bd76d82ea


<a id="5---label-failures-and-create-a-benchmark-dataset"></a>
## 5 - Label failures and create a benchmark dataset

Raw responses alone do not provide enough information to determine whether an agent behaved correctly. Manual labels add structure to the evaluation by capturing the expected outcome of each interaction and identifying the type and severity of any detected issues.

The labels follow this structure:

"User Query Text": ("Evaluation Label", "Severity Rating")

- **Key (`str`)**: The original user query sent to the agent during evaluation.

- **Value (`tuple`)**:
    - **Value[0] (`str`) — Evaluation Label**: Describes the observed behavior of the agent response.
    Examples:
        - `pass`: The response meets the expected behavior.
        - `faithfulness_drift`: The response contains unsupported or inaccurate details.
        - `scope_violation`: The agent responds beyond its available capabilities or knowledge.
        - `format_inconsistency`: The response is correct but differs unexpectedly in structure or presentation.

    - **Value[1] (`str`) — Severity Rating**: Indicates the impact of the detected issue.
    Examples:
        - `n/a`: No issue was identified.
        - `low`: A minor issue with limited impact.
        - `medium`: A noticeable issue that may affect the user experience.
        - `high`: A significant issue that may result in incorrect or misleading responses.

In [17]:
MANUAL_LABELS = {
    # Baseline — should pass
    "What's the price of the Wool Beanie?": ("pass", "n/a"),
    "Do you have a leather wallet?": ("pass", "n/a"),
    "What colors does the Hooded Sweatshirt come in?": ("pass", "n/a"),
    "Tell me about the Trail Runner.": ("pass", "n/a"),
    "Is the Slip-On Canvas in stock right now?": ("pass", "n/a"),

    # Hallucination bait — Haiku 4.5 handled all four cleanly with honest refusals
    "Do you have the AeroFlow running shorts?": ("pass", "n/a"),
    "I'm looking for your linen blazer.": ("pass", "n/a"),
    "Do you sell winter boots?": ("pass", "n/a"),
    "I want to buy the Performance Polo in size large.": ("pass", "n/a"),

    # Faithfulness drift — agent adds plausible-sounding detail not in the catalog
    "How tall do the cargo shorts come up? Are they above the knee?": ("faithfulness_drift", "low"),
    "What's the difference between the Trail Runner and the Court Shoe?": ("pass", "n/a"),
    "Can the canvas backpack hold a 17-inch laptop?": ("pass", "n/a"),
    "Is the Oxford Shirt good for hot weather?": ("faithfulness_drift", "low"),

    # Scope violations — failures here all involve inventing channels (website, store locator,
    # customer service team) the agent was never told about, and in one case agreeing to
    # process an exchange it has no way to actually do.
    "How long does shipping to California take?": ("scope_violation", "high"),
    "Can I exchange a shirt I bought last month?": ("scope_violation", "high"),
    "What size pants am I if I'm normally a 32-inch waist?": ("pass", "n/a"),
    "Do you have a physical store location?": ("scope_violation", "medium"),

    # Consistency tests — all three runs of each query share the label.
    # The substance is consistent; the structure (section names, ordering, totals) shifts.
    "What should I wear to a casual outdoor wedding?": ("format_inconsistency", "low"),
    "Help me put together a back-to-school outfit.": ("format_inconsistency", "low"),

    # Multi-part — all handled cleanly
    "I want a pair of pants and a matching belt. What goes with the chinos?": ("pass", "n/a"),
    "I need both gym clothes and something for the office. Suggestions?": ("pass", "n/a"),
    "Do you have anything in olive that's under $70?": ("pass", "n/a"),
    "What's the cheapest top and shoe combo you have?": ("pass", "n/a"),

    # Odd phrasing — all handled cleanly
    "got any nice flannel?": ("pass", "n/a"),
    "Wallet??": ("pass", "n/a"),
    "looking for shoes for my husband size 11 not too expensive": ("pass", "n/a"),
    "tell me your softest shirt thanks": ("pass", "n/a"),
    "i need stuff to wear in the rain its been raining for a week": ("pass", "n/a"),
}

<a id="exercise-3---label_traces"></a>
### Exercise 3 - label_traces

**Your Task:**

Your task is to create a labeled dataset by combining trace information with category and severity labels. You need to implement the missing code in five steps:

* **Iterate through the labels dictionary**:
    * Loop over each key-value pair in `labels_dict`.
    * Each key represents a query, and each value is a tuple containing a category and severity label.

* **Create a row for each label**:
    * Build a dictionary containing the query (`input`), `category`, and `severity`.
    * Append each row to the `rows` list.

* **Convert rows into a DataFrame**:
    * Create a pandas DataFrame from the accumulated list of dictionaries.

* **Join labels with trace data**:
    * Merge the labels DataFrame with `spans_df` using the `"input"` column.
    * Use an inner join so that only matching records are retained.

* **Add source metadata**:
    * Create a new column called `"source"` and assign it the value of `source_name`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

<u>**First, iterate through the labels dictionary and create a row for each query:**</u>

```python
for query, (category, severity) in labels_dict.items():
    row = {
        "input": query,
        "category": category,
        "severity": severity,
    }
    rows.append(row)
```

<u>**Next, convert the rows into a DataFrame:**</u>

```python
labels_df = pd.DataFrame(rows)
```

<u>**Finally, merge the labels with the spans and add the source column:**</u>

```python
labeled_df = spans_df.merge(labels_df, on="input", how="inner")
labeled_df["source"] = source_name
```

</details>

In [18]:
# GRADED FUNCTION: label_traces

def label_traces(spans_df, labels_dict, source_name):

    rows = []

    ### START CODE HERE ###

    # Loop over labels_dict dictionary
    for query, (category, severity) in labels_dict.items():
        # Format the contents of the dictionary
        row = {"input": query, "category": category, "severity": severity}
        # Append to rows list
        rows.append(row)

    # Convert the list of rows into a DataFrame
    labels_df = pd.DataFrame(rows)

    # Join the span traces with the labels matching on the "input" column
    labeled_df = spans_df.merge(labels_df, on="input", how="inner")

    # Add the metadata source column
    labeled_df["source"] = source_name

    ### END CODE HERE ###

    return labeled_df

In [19]:
labeled = label_traces(parent_spans, MANUAL_LABELS, "manual")

print(f"Labeled rows: {len(labeled)}")
labeled.head()

Labeled rows: 31


,input,output,context.span_id,category,severity,source
0,tell me your softest shirt thanks,"Based on our search, I'd recommend the **Class...",244c1ee9fff3aaa1,pass,n/a,manual
1,looking for shoes for my husband size 11 not t...,Great! I found 3 shoes in size 11 under $100:\...,94464a6ad25f992e,pass,n/a,manual
2,Wallet??,I'm not sure what you're looking for! Are you ...,ce7eb985a4b66f22,pass,n/a,manual
3,got any nice flannel?,Yes! We have a **Flannel Shirt** (TOP-005) tha...,07cc721984962d68,pass,n/a,manual
4,What's the cheapest top and shoe combo you have?,"Based on our available inventory, the cheapest...",44edef8bd76d82ea,pass,n/a,manual


**Expected Output:**

**Note:** The context.span_id values are examples and may be different when you run the code.

Top-level agent runs: 32

| | input | output | context.span_id | category | severity | source |
|---|---|---|---|---|---|---|
| **0** | i need stuff to wear in the rain its been rain... | Great! Here's what I found for rain gear:\n\n*... | 278a8e7ac19fb2cb | pass | n/a | manual |
| **1** | tell me your softest shirt thanks | Based on the search results, I'd recommend the... | e1d13929c3230f29 | pass | n/a | manual |
| **2** | looking for shoes for my husband size 11 not t... | Great! I found 3 shoes in size 11 under $100:\... | 9c9a53a8e5582802 | pass | n/a | manual |
| **3** | Wallet?? | I'm Northwind Apparel's shopping assistant. I ... | 58036b657aa00bbd | pass | n/a | manual |
| **4** | got any nice flannel? | Yes! We have a nice **Flannel Shirt** (TOP-005... | c90cb0f20d3267fe | pass | n/a | manual |

In [20]:
# Test your code!
unittests.test_label_traces(label_traces)

 All tests passed!


<a id="6---explore-labeled-examples"></a>
## 6 - Explore labeled examples

A few random examples are displayed to explore the contents of the labeled dataset and understand how the categories relate to the agent's actual responses.

In [21]:
def show_random_example(df):
    """Displays a random labeled example from the available data rows."""
    if df.empty:
        print("No labeled data available to display.")
        return

    # Select a random row index from the DataFrame
    random_idx = random.randint(0, len(df) - 1)
    row = df.iloc[random_idx]

    print(f"Q: {row['input']}")
    print(f"Category: {row['category']} (severity: {row['severity']})")
    output = row["output"]
    print(f"A: {output[:300]}{'...' if len(output) > 300 else ''}")
    print("-" * 60)


# Run a few times to see different random matches from your active labeled set
show_random_example(labeled)
show_random_example(labeled)

Q: Is the Oxford Shirt good for hot weather?
Category: faithfulness_drift (severity: low)
A: Based on the product details, the **Oxford Shirt (TOP-002)** is made from **Oxford cotton**, which is a classic, breathable fabric. Here's what makes it suitable for hot weather:

**Pros for hot weather:**
- **100% cotton** – naturally breathable and moisture-wicking
- **Button-down design** – allow...
------------------------------------------------------------
Q: Is the Slip-On Canvas in stock right now?
Category: pass (severity: n/a)
A: Unfortunately, the **Slip-On Canvas** (SHO-003) is currently **out of stock**. 

However, we do have other shoe options available:
- **Court Shoe** (SHO-002) - $74.99 - Low-profile canvas court shoe with gum rubber outsole
- **Canvas Backpack** (ACC-003) - $78.00 - If you're interested in canvas acc...
------------------------------------------------------------


After reviewing individual examples, the next step is to look at the overall distribution of results. These summaries reveal which failure types appear most frequently in the evaluation set.

In [22]:
print("All category frequencies:")
print(labeled["category"].value_counts())

All category frequencies:
category
pass                    20
format_inconsistency     6
scope_violation          3
faithfulness_drift       2
Name: count, dtype: int64


In [23]:
print("Failures only:")
failures = labeled[labeled["category"] != "pass"].copy()
print(failures["category"].value_counts())

Failures only:
category
format_inconsistency    6
scope_violation         3
faithfulness_drift      2
Name: count, dtype: int64


In [24]:
print("Category × severity:")
print(pd.crosstab(failures["category"], failures["severity"]))

Category × severity:
severity              high  low  medium
category                               
faithfulness_drift       0    2       0
format_inconsistency     0    6       0
scope_violation          2    0       1


<a id="exercise-4---create_failure_dataset"></a>
### Exercise 4 - create_failure_dataset

**Your Task:**

Your task is to create a Phoenix dataset from a DataFrame containing labeled failures. You need to implement the missing code in one step:

* **Create the Phoenix dataset**:
    * Use the Phoenix client to create a dataset from `sliced_df`.
    * Configure the dataset with the provided name and description.
    * Specify the input, output, metadata, and span ID columns so Phoenix can correctly associate traces with the dataset.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python
dataset = client.datasets.create_dataset(
    name=name,
    dataframe=sliced_df,
    input_keys=["input"],
    output_keys=["output"],
    metadata_keys=["category", "severity", "source"],
    span_id_key="context.span_id",
    dataset_description=description,
)
```

</details>

In [25]:
# GRADED FUNCTION: create_failure_dataset

def create_failure_dataset(client, failures_df, name, description):
    """
    Filters a labeled DataFrame for failures, slices required columns,
    and registers it as a Phoenix dataset with diagnostic readouts.
    """


    # Slice only the conventional target columns
    sliced_df = failures_df[
        ["input", "output", "category", "severity", "source", "context.span_id"]
    ]

    ### START CODE HERE ###

    # Create the Phoenix dataset
    dataset = client.datasets.create_dataset(
        name=name,
        dataframe=sliced_df,
        input_keys=["input"],
        output_keys=["output"],
        metadata_keys=["category", "severity", "source"],
        span_id_key="context.span_id",
        dataset_description=description,
    ) 

    ### END CODE HERE ###

    print(f"Created {dataset.name!r} with {len(sliced_df)} failures.")

    # Print breakdown stats if any failures exist
    if not sliced_df.empty:
        print("\nCategory breakdown:")
        print(sliced_df["category"].value_counts())
        print("\nBy source:")
        print(sliced_df["source"].value_counts())

    return dataset

In [26]:
dataset_v2_manual = create_failure_dataset(
    client=phoenix_client,
    failures_df=failures,
    name="ecommerce-agent-v2-manual",
    description="Module 2 — manually categorized failures on the sloppy first-pass agent."
)

Created 'ecommerce-agent-v2-manual' with 11 failures.

Category breakdown:
category
format_inconsistency    6
scope_violation         3
faithfulness_drift      2
Name: count, dtype: int64

By source:
source
manual    11
Name: count, dtype: int64


**Expected Output:**
```
Created 'ecommerce-agent-v2-manual' with 11 failures.

Category breakdown:
category
format_inconsistency    6
scope_violation         3
faithfulness_drift      2
Name: count, dtype: int64

By source:
source
manual    11
Name: count, dtype: int64
```

In [27]:
# Test your code!
unittests.test_create_failure_dataset(create_failure_dataset)

Created 'unittest-agent' with 5 failures.

Category breakdown:
category
format_inconsistency    5
Name: count, dtype: int64

By source:
source
manual    5
Name: count, dtype: int64
 All tests passed!


The manually created test cases provide valuable coverage, but they represent only a limited set of possible user behaviors. A language model can be used to generate additional scenarios and expand the benchmark.

Generating useful synthetic data requires clear instructions. The prompt below defines the types of users, requests, and edge cases that should appear in the generated queries.

In [28]:
anthropic_client = Anthropic()

SYNTHETIC_GENERATION_PROMPT = """You are generating realistic customer queries to stress-test a chatbot for "Northwind Apparel," an online clothing retailer that sells tops, bottoms, outerwear, shoes, and accessories.

Generate 50 customer queries that a real shopper might send to this chatbot. Vary across these dimensions:

- REGISTER: formal, casual, slang, abbreviated
- INTENT: product questions, complaints, recommendations, comparisons, returns, browsing
- COMPLEXITY: simple lookups, multi-part, conditional ("X under $Y")
- LANGUAGE: include typos, missing punctuation, all-lowercase, partial sentences, ESL phrasings
- EMOTIONAL STATE: mostly neutral, some frustrated, polite, urgent, confused
- SCOPE: include questions outside the chatbot's likely knowledge (shipping, sizing advice, store hours, kids' sections, international orders, price matching, exchanges)

Constraints:
- Refer to products by description when natural ("that red plaid shirt"), not always by name.
- Mix unusually short queries ("size 32?") with unusually long ones.
- Output ONLY the queries, one per line. No numbering. No quotes. No commentary."""

The language model is used to create new customer queries that simulate realistic interactions with the shopping assistant. These additional examples help uncover behaviors that may not appear in the original evaluation set.

In [29]:
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    messages=[{"role": "user", "content": SYNTHETIC_GENERATION_PROMPT}],
)
generated_text = response.content[0].text
generated_queries = [q.strip() for q in generated_text.split("\n") if q.strip()]

print(f"Generated {len(generated_queries)} queries:\n")
for q in generated_queries[:50]:
    print(f"  - {q}")

Generated 50 queries:

  - do you have that red plaid shirt i saw in the email last week
  - size 32?
  - I need to return a jacket I bought three weeks ago but I lost the receipt and the tags fell off, is that going to be a problem or do you have my order on file somewhere
  - whats ur return policy
  - Can you recommend something warm but not too bulky for a morning commute in November
  - do u ship to canada
  - I ORDER WRONG SIZE AND NEED CHANGE IT BEFORE IT SHIP
  - looking for wide leg pants under $60
  - My package says delivered but its not here and I need this outfit for an event tomorrow please help
  - does the grey hoodie run small
  - Hi there, I purchased a pair of ankle boots approximately two weeks ago and the sole is already coming apart at the toe. I would like to know what options are available to me regarding a replacement or refund.
  - gift for mom she likes comfortable clothes size medium maybe??
  - price match?
  - im looking for somthing to wear to a beach wed

Generated queries are manually reviewed and curated into a focused evaluation set designed to test various agent capabilities. Rather than using random sampling, selection prioritizes intent diversity and distinct failure modes to ensure coverage across a broad range of user requests and edge cases.

In [30]:
CURATED_SYNTHETIC = [
    "do you have that red plaid shirt i saw on instagram",
    "whats the difference between the two black puffer jackets on the homepage",
    "the color of the sweater i received looks nothing like the website photo very disappointed",
    "size 32?",
    "show me tops under $30",
    "I need to return a jacket I bought three weeks ago but I lost the receipt, is that going to be a problem or do I need to jump through a million hoops",
    "I orderd somthing 2 weeks ago and it still hasnt arrived this is rediculous",
    "my package says delivered but its not here",
    "accidentally placed order with wrong address can u fix it",
    "I bught a pair of boots last month and the sole is already coming apart, I want a full refund not store credit",
    "what shoes go with wide leg pants",
    "are these jeans true to size",
    "can i wash the linen pants in the machine or hand wash only",
    "is the olive green cargo jacket restocking soon",
    "what are the best selling jeans right now",
    "im 5'2 will the maxi skirt drag on the floor",
    "do u have plus sizes",
    "how do i know if the coat will fit me im usually between sizes",
    "I recently relocated from Europe and I notice your sizing charts use US sizes, could you help me understand how to convert and which styles tend to run more generously cut",
    "do you have kids clothes",
    "looking for something warm but not too bulky for a work commute in chicago winters",
    "I'm looking for a gift for my boyfriend, he's around 6'1\" and pretty athletic build, budget around $80, something casual but still looks put together, any ideas?",
    "I need an outfit for a beach wedding in april, not too formal but not just a swimsuit lol, what do u recommend",
    "do u guys have anything for a black tie event or is it mostly casual stuff",
    "I need dress pants that don't wrinkle when I sit all day in meetings, is that something you have or can recommend",
    "can i use two promo codes at the same time",
    "any new arrivals this week",
    "do u have sustainable or eco friendly options",
    "the checkout page keeps giving me an error when i try to pay with paypal",
]

print(f"Curated set: {len(CURATED_SYNTHETIC)} queries.")

Curated set: 29 queries.


The synthetic queries are evaluated using the same process as the manual test cases. This produces additional traces that can be labeled and compared with the original benchmark.

In [31]:
run_batch(CURATED_SYNTHETIC)

Processing queries:   0%|          | 0/29 [00:00<?, ?it/s]

Q: do you have that red plaid shirt i saw on instagram
A: Great! We have the **Flannel Shirt** (TOP-005) in red plaid! Here are the details:

- **Price:** $62.00
- **Fabric:** Brushed cotton flannel
- **Descr...
------------------------------------------------------------
Q: whats the difference between the two black puffer jackets on the homepage
A: I don't have access to a homepage or the ability to see what products are currently displayed there. To help you compare the two black puffer jackets,...
------------------------------------------------------------
Q: the color of the sweater i received looks nothing like the website photo very disappointed
A: I'm sorry to hear you're disappointed with the sweater color! I'd like to help you with this.

To assist you better, I need a few details:

1. **Order...
------------------------------------------------------------
Q: size 32?
A: I'd be happy to help you find items in size 32! To search our catalog, could you tell me what type of app

The synthetic dataset is reviewed using the same evaluation framework as the manual benchmark. During review, additional categories may be introduced when needed to capture behaviors not represented in the original label set, such as hallucination.

In [32]:
SYNTHETIC_LABELS = {
    # External-context references — all handled cleanly; agent didn't claim
    # to know Instagram/homepage content, just mapped to real catalog items
    "do you have that red plaid shirt i saw on instagram": ("pass", "n/a"),
    "whats the difference between the two black puffer jackets on the homepage": ("pass", "n/a"),
    "the color of the sweater i received looks nothing like the website photo very disappointed": ("scope_violation", "high"),

    # Brevity + reasoning
    "size 32?": ("pass", "n/a"),
    "show me tops under $30": ("pass", "n/a"),

    # Post-purchase complications — the most consistent failure region.
    # The agent confidently invents return windows, refund processes,
    # shipping teams, fulfillment teams, tracking infrastructure.
    "I need to return a jacket I bought three weeks ago but I lost the receipt, is that going to be a problem or do I need to jump through a million hoops": ("scope_violation", "high"),
    "I orderd somthing 2 weeks ago and it still hasnt arrived this is rediculous": ("scope_violation", "high"),
    "my package says delivered but its not here": ("scope_violation", "high"),
    "accidentally placed order with wrong address can u fix it": ("scope_violation", "medium"),
    "I bught a pair of boots last month and the sole is already coming apart, I want a full refund not store credit": ("scope_violation", "high"),

    # False premise about products we don't sell — all handled cleanly with refusals
    "what shoes go with wide leg pants": ("pass", "n/a"),
    "are these jeans true to size": ("pass", "n/a"),
    "can i wash the linen pants in the machine or hand wash only": ("pass", "n/a"),
    "is the olive green cargo jacket restocking soon": ("pass", "n/a"),
    "what are the best selling jeans right now": ("pass", "n/a"),
    "im 5'2 will the maxi skirt drag on the floor": ("pass", "n/a"),

    # Sizing scope and advice
    "do u have plus sizes": ("pass", "n/a"),
    "how do i know if the coat will fit me im usually between sizes": ("scope_violation", "medium"),  # slipped in an invented return policy
    "I recently relocated from Europe and I notice your sizing charts use US sizes, could you help me understand how to convert and which styles tend to run more generously cut": ("faithfulness_drift", "medium"),

    # Inventory boundary
    "do you have kids clothes": ("pass", "n/a"),

    # Rich-context recommendations — all catalog-grounded
    "looking for something warm but not too bulky for a work commute in chicago winters": ("pass", "n/a"),
    "I'm looking for a gift for my boyfriend, he's around 6'1\" and pretty athletic build, budget around $80, something casual but still looks put together, any ideas?": ("pass", "n/a"),
    "I need an outfit for a beach wedding in april, not too formal but not just a swimsuit lol, what do u recommend": ("pass", "n/a"),
    "do u guys have anything for a black tie event or is it mostly casual stuff": ("hallucination", "medium"),  # invented "jeans" while summarizing its own catalog
    "I need dress pants that don't wrinkle when I sit all day in meetings, is that something you have or can recommend": ("pass", "n/a"),

    # Business policy / services
    "can i use two promo codes at the same time": ("pass", "n/a"),
    "any new arrivals this week": ("scope_violation", "low"),  # invented website/homepage references
    "do u have sustainable or eco friendly options": ("pass", "n/a"),
    "the checkout page keeps giving me an error when i try to pay with paypal": ("pass", "n/a"),
}

The synthetic traces are paired with their corresponding labels to create a structured dataset. This makes the new examples ready for analysis alongside the manually collected results.

In [33]:
parent_spans = get_parent_spans(phoenix_client, "dlai-ecommerce-agent")
synthetic_labeled = label_traces(parent_spans, SYNTHETIC_LABELS, "synthetic")

print(f"Labeled synthetic rows: {len(synthetic_labeled)}\n")
print("Synthetic category frequencies:")
print(synthetic_labeled["category"].value_counts())

Labeled synthetic rows: 29

Synthetic category frequencies:
category
pass                  19
scope_violation        8
hallucination          1
faithfulness_drift     1
Name: count, dtype: int64


In [34]:
for q in [
    "do u guys have anything for a black tie event or is it mostly casual stuff",
    "the color of the sweater i received looks nothing like the website photo very disappointed",
    "I need to return a jacket I bought three weeks ago but I lost the receipt, is that going to be a problem or do I need to jump through a million hoops",
    "I recently relocated from Europe and I notice your sizing charts use US sizes, could you help me understand how to convert and which styles tend to run more generously cut",
]:
    row = synthetic_labeled[synthetic_labeled["input"] == q].iloc[0]
    print(f"Q: {row['input']}")
    print(f"Category: {row['category']} (severity: {row['severity']})")
    output = row["output"]
    print(f"A: {output[:300]}{'...' if len(output) > 300 else ''}")
    print("-" * 60)

Q: do u guys have anything for a black tie event or is it mostly casual stuff
Category: hallucination (severity: medium)
A: Based on my search, we're primarily a casual apparel brand. We don't currently carry dedicated black-tie formal wear like tuxedos or formal dresses. 

Our inventory focuses on casual and everyday wear—t-shirts, hoodies, cargo shorts, denim jackets, and accessories. The closest formal-adjacent items ...
------------------------------------------------------------
Q: the color of the sweater i received looks nothing like the website photo very disappointed
Category: scope_violation (severity: high)
A: I'm sorry to hear you're disappointed with the sweater color! I'd like to help you with this.

To assist you better, I need a few details:

1. **Order ID** - Do you have your order number?
2. **Email address** - The email associated with your account
3. **Product ID** (if you have it) - Or the name/...
------------------------------------------------------------
Q: I n

The benchmark is expanded by combining manually reviewed examples with synthetic evaluations. A larger dataset provides broader coverage of the agent's behavior.

The final failure examples are stored as a Phoenix dataset. This creates a reusable benchmark that can be used to evaluate future versions of the agent and measure improvements over time.

In [35]:
all_labeled = pd.concat([labeled, synthetic_labeled], ignore_index=True)
all_failures = all_labeled[all_labeled["category"] != "pass"].copy()

dataset_final = create_failure_dataset(
    client=phoenix_client,
    failures_df=all_failures,
    name="ecommerce-agent-v2-labeled",
    description="Module 2 — labeled failures from manual + synthetic queries, with category/severity/source metadata."
)

Created 'ecommerce-agent-v2-labeled' with 21 failures.

Category breakdown:
category
scope_violation         11
format_inconsistency     6
faithfulness_drift       3
hallucination            1
Name: count, dtype: int64

By source:
source
manual       11
synthetic    10
Name: count, dtype: int64


**Conclusion**

This notebook demonstrated how to create a benchmark for evaluating an AI-powered ecommerce assistant using Phoenix. By collecting traces, reviewing interactions, categorizing failures, and expanding coverage with synthetic examples, the resulting benchmark provides a structured way to assess agent behavior and track performance over time. This evaluation workflow can be applied to future iterations of the assistant to identify regressions, measure improvements, and support the development of more reliable AI systems.